In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

import utils
import vision_transformer as vits
from vision_transformer import DINOHead
from functools import partial

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

device: mps


In [20]:
EPOCHS = 50
BATCH_SIZE = 32
LR = 5e-4 * BATCH_SIZE / 256
GLOBAL_SIZE, LOCAL_SIZE = 96, 48   # small crops to keep it fast (paper uses 224 / 96)
N_LOCAL = 8                       # number of small local crops
OUT_DIM = 4096                    # DINO head output dim (paper uses 65536)
TEACHER_MOMENTUM = 0.996
TEACHER_TEMP, STUDENT_TEMP = 0.04, 0.1

In [21]:
normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
color = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
])

class MultiCrop:
    def __init__(self):
        self.global_t = transforms.Compose([
            transforms.RandomResizedCrop(GLOBAL_SIZE, scale=(0.4, 1.0)), color,
            utils.GaussianBlur(0.5), normalize])
        self.local_t = transforms.Compose([
            transforms.RandomResizedCrop(LOCAL_SIZE, scale=(0.05, 0.4)), color,
            utils.GaussianBlur(0.5), normalize])

    def __call__(self, img):
        img = img.convert("RGB")  # some images are grayscale
        return [self.global_t(img), self.global_t(img)] + [self.local_t(img) for _ in range(N_LOCAL)]

train_set = datasets.ImageFolder("data/train", transform=MultiCrop())
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                                           num_workers=0, drop_last=True)
print(len(train_set), "train images,", len(train_set.classes), "classes (unused for training)")

2400 train images, 16 classes (unused for training)


In [29]:
def build():
    backbone = vits.VisionTransformer(
        patch_size=16, img_size=[GLOBAL_SIZE], embed_dim=192, depth=6, num_heads=3,
        mlp_ratio=4, qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6))
    return utils.MultiCropWrapper(backbone, DINOHead(backbone.embed_dim, OUT_DIM))

student = build().to(device)
teacher = build().to(device)
teacher.load_state_dict(student.state_dict())  # start teacher = student
for p in teacher.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.04)

In [27]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, center_momentum=0.9):
        super().__init__()
        self.center_momentum = center_momentum
        self.register_buffer("center", torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out):
        s = (student_out / STUDENT_TEMP).chunk(2 + N_LOCAL)
        t = F.softmax((teacher_out - self.center) / TEACHER_TEMP, dim=-1).detach().chunk(2)
        loss, n = 0, 0
        for i, q in enumerate(t):
            for v, p in enumerate(s):
                if v == i:
                    continue  # skip same view
                loss += torch.sum(-q * F.log_softmax(p, dim=-1), dim=-1).mean()
                n += 1
        self.update_center(teacher_out)
        return loss / n

    @torch.no_grad()
    def update_center(self, teacher_out):
        batch_center = teacher_out.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

dino_loss = DINOLoss(OUT_DIM).to(device)

In [ ]:
history = []
for epoch in range(EPOCHS):
    student.train()
    total = 0.0
    for crops, _ in train_loader:
        crops = [c.to(device) for c in crops]
        teacher_out = teacher(crops[:2])   # teacher only sees the 2 global crops
        student_out = student(crops)       # student sees all crops
        loss = dino_loss(student_out, teacher_out)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA update of the teacher
        with torch.no_grad():
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.mul_(TEACHER_MOMENTUM).add_(ps.detach(), alpha=1 - TEACHER_MOMENTUM)
        total += loss.item()

    history.append(total / len(train_loader))
    print(f"epoch {epoch + 1}/{EPOCHS}  loss {history[-1]:.4f}")



In [ ]:
torch.save(teacher.backbone.state_dict(), "Models/dino_vit_tiny_teacher.pth")

In [25]:
eval_t = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.Resize(GLOBAL_SIZE), transforms.CenterCrop(GLOBAL_SIZE), normalize])

@torch.no_grad()
def extract(folder):
    ds = datasets.ImageFolder(folder, transform=eval_t)
    feats, labels = [], []
    for x, y in torch.utils.data.DataLoader(ds, batch_size=64):
        feats.append(F.normalize(teacher.backbone(x.to(device)), dim=1).cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

teacher.eval()
train_f, train_y = extract("data/train")
test_f, test_y = extract("data/test")

k = 20
sim = test_f @ train_f.T
topk = sim.topk(k, dim=1).indices
pred = train_y[topk].mode(dim=1).values
print(f"{k}-NN test accuracy: {(pred == test_y).float().mean():.3f}  (chance = {1/len(train_set.classes):.3f})")

20-NN test accuracy: 0.365  (chance = 0.062)
